# Daily Challenge : MCP Weather (Version Étudiant - CORRIGÉ)

Serveur + client MCP simples, adaptés aux débutants (sans LLM). Le code ci-dessous est complet et commenté en français.

## Installation
Exécutez la cellule d'installation ci-dessous. Si Colab vous le demande, redémarrez le runtime après l'installation.

In [ ]:
# On installe le SDK MCP ainsi que son outil en ligne de commande (CLI)
%pip install -qU "mcp[cli]"

In [ ]:
# Vérification rapide de l'installation
!python --version
!mcp --help | head -n 5

## A) Le serveur (server.py)

On implémente le serveur `WeatherDemo` avec :
- Un **outil** `get_weather(city)` qui renvoie la météo pour une ville supportée.
- Une **ressource** `cities://list` qui renvoie la liste des villes supportées, **une par ligne**.

In [ ]:

%%writefile server.py
# ============================================================
# server.py
# ------------------------------------------------------------
# Serveur MCP "WeatherDemo" : expose un outil météo et une ressource
# listant les villes supportées. Aucune donnée réelle n'est utilisée,
# c'est un petit dictionnaire statique en mémoire (pour l'apprentissage).
# ============================================================

import logging
from mcp.server.fastmcp import FastMCP

# On configure la journalisation (logging) pour afficher les infos sur stderr.
# Cela permet de voir dans le terminal quand un outil est appelé.
logging.basicConfig(level=logging.INFO)

# On crée le serveur MCP et on le nomme "WeatherDemo"
mcp = FastMCP("WeatherDemo")

# Petite "base de données" en mémoire : un dictionnaire associant
# chaque ville (en minuscules) à ses données météo
CITY_DATA = {
    "paris": {"temp_c": 21, "condition": "sunny"},
    "london": {"temp_c": 18, "condition": "cloudy"},
    "new york": {"temp_c": 24, "condition": "breezy"},
}


# ------------------------------------------------------------
# OUTIL (TOOL) : get_weather
# ------------------------------------------------------------
# Le décorateur @mcp.tool() enregistre cette fonction comme un outil
# que le client pourra découvrir et appeler à distance.
@mcp.tool()
def get_weather(city: str) -> dict:
    """Retourne des infos météo simples pour une ville supportée."""
    # On normalise le nom de la ville : on enlève les espaces superflus
    # et on met tout en minuscules, pour que "Paris" et "paris" soient équivalents
    key = city.strip().lower()

    # On cherche la ville dans notre dictionnaire
    data = CITY_DATA.get(key)

    # Si la ville n'existe pas dans notre base, on retourne un dict d'erreur
    # (plutôt que de faire planter le serveur)
    if not data:
        return {"error": f"Unsupported city: {city}. Try one of: {', '.join(CITY_DATA)}"}

    # On journalise l'appel (visible côté serveur, utile pour déboguer)
    logging.info("get_weather called for %s", key)

    # On retourne un dictionnaire combinant le nom de la ville et ses données météo
    # ("**data" "déplie" le dictionnaire data à l'intérieur du nouveau dictionnaire)
    return {"city": key, **data}


# ------------------------------------------------------------
# RESSOURCE (RESOURCE) : cities://list
# ------------------------------------------------------------
# Une ressource est une donnée en lecture seule, accessible via une URI fixe.
# Ici, l'URI "cities://list" ne contient pas de variable : elle retourne
# toujours la même chose (la liste des villes supportées).
@mcp.resource("cities://list")
def list_cities() -> str:
    """Retourne les villes supportées sous forme de texte, une par ligne."""
    # IMPORTANT : on sépare les villes par un retour à la ligne "\n"
    # (et non par une chaîne vide, sinon les noms seraient collés ensemble)
    return "\n".join(sorted(CITY_DATA))


# ------------------------------------------------------------
# POINT D'ENTRÉE DU PROGRAMME
# ------------------------------------------------------------
if __name__ == "__main__":
    # On démarre la boucle du serveur.
    # Par défaut, FastMCP utilise le transport STDIO (entrée/sortie standard)
    mcp.run()


## B) Le client (client.py)

Le client démarre le serveur via STDIO grâce à la commande `mcp`, découvre ses capacités (ressources et outils), puis les utilise.

In [ ]:

%%writefile client.py
# ============================================================
# client.py
# ------------------------------------------------------------
# Ce client MCP :
#   1. Démarre le serveur (server.py) via la commande CLI "mcp run"
#   2. Initialise une session
#   3. Liste les ressources et les outils disponibles
#   4. Lit la ressource "cities://list"
#   5. Appelle l'outil "get_weather" pour la ville "Paris"
# ============================================================

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# On définit comment lancer le serveur : la commande "mcp run server.py"
# sera exécutée automatiquement par le client via STDIO.
server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


def extract_content(payload):
    """Fonction utilitaire : essaie d'extraire le contenu utile d'une réponse MCP."""
    # Cas 1 : la réponse contient une liste "contents" (ex: lecture de ressource)
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    # Cas 2 : la réponse contient directement un attribut "content" (ex: appel d'outil)
    if hasattr(payload, "content"):
        return payload.content
    # Cas 3 (secours) : on retourne la représentation texte brute
    return str(payload)


async def run():
    # On ouvre la connexion STDIO ; le serveur est lancé automatiquement
    async with stdio_client(server_params) as (read, write):
        # On crée la session cliente à partir des flux de lecture/écriture
        async with ClientSession(read, write) as session:
            # Étape obligatoire : on initialise la session avec le serveur
            await session.initialize()

            # --------------------------------------------------------
            # 1) Liste des ressources disponibles
            # --------------------------------------------------------
            resources = await session.list_resources()
            print("Resources:")
            for res in resources.resources:
                print(f"- {res.uri} ({res.name or ''})")

            # --------------------------------------------------------
            # 2) Liste des outils disponibles
            # --------------------------------------------------------
            tools = await session.list_tools()
            print("Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}")

            # --------------------------------------------------------
            # 3) Lecture de la ressource "cities://list"
            # --------------------------------------------------------
            cities = await session.read_resource("cities://list")
            print("cities://list ->")
            print(extract_content(cities))

            # --------------------------------------------------------
            # 4) Appel de l'outil "get_weather" pour la ville "Paris"
            # --------------------------------------------------------
            weather = await session.call_tool("get_weather", {"city": "Paris"})
            print("get_weather(Paris) ->", extract_content(weather))


if __name__ == "__main__":
    # On lance la fonction asynchrone run() dans la boucle d'événements asyncio
    asyncio.run(run())


## C) Exécution

**Un seul terminal** (le client lance automatiquement le serveur) :
```bash
python client.py
```

**Deux terminaux** (pour déboguer séparément) :
```bash
# Terminal 1
mcp run server.py

# Terminal 2
python client.py
```

Dans Colab, exécutez simplement la cellule suivante.

In [ ]:
# On exécute le client (qui démarre le serveur via STDIO automatiquement)
!python client.py

## Dépannage (Troubleshooting)

- `mcp: command not found` → relancez la cellule d'installation ou activez votre venv.
- Aucun outil/ressource listé → vérifiez les décorateurs (`@mcp.tool()`, `@mcp.resource(...)`) et redémarrez le serveur.
- Ville manquante → utilisez l'une des villes supportées, listées via la ressource `cities://list` (paris, london, new york).